# 🗺️ The Three Ways to Customize Agent Behavior

## Wrap-style Middleware vs Node-style Middleware vs Tool Calls

---

![Middleware Comparison Table](resources/middleware_comparison_table.svg)

---

LangChain gives you **three distinct mechanisms** to customize how an agent behaves at runtime. They look similar but operate at different levels and serve different purposes.

| | 🔀 Wrap-style Middleware | 🪝 Node-style Middleware | 🔧 Tool Calls |
|---|---|---|---|
| **Receives** | `ModelRequest` | `State` + `Runtime` | `ToolRuntime` |
| **Who controls it** | You (the developer) | You (the developer) | The LLM (the agent itself) |
| **What it adjusts** | Tools, prompt, model | State (messages, custom fields) | State or retrieves context |
| **When it runs** | Around each LLM/tool call | Before/after agent or tool nodes | When the LLM decides to call a tool |
| **Example** | Give internal users extra tools | Trim old messages | Update state with username |

### 🔑 The fundamental distinction

- **Middleware** (wrap + node) = **you** decide what happens, based on rules you write. The LLM doesn't know middleware exists.
- **Tool calls** = the **LLM** decides what happens. It chooses to call a tool based on the conversation.

Think of it this way:
- Middleware = **guardrails and plumbing** (invisible to the LLM)
- Tools = **capabilities** (visible to the LLM, it chooses when to use them)

---

## 📐 When to Use Which?

| I want to... | Use | Why |
|---|---|---|
| Swap models based on user tier | 🔀 `wrap_model_call` | Developer decision, LLM shouldn't know |
| Filter tools by user role | 🔀 `wrap_model_call` | Security policy, not LLM's choice |
| Generate prompts per-user | 🔀 `dynamic_prompt` | Developer controls the prompt |
| Trim old messages | 🪝 `@before_agent` | State management, invisible to LLM |
| Summarize conversation | 🪝 `SummarizationMiddleware` | State management, invisible to LLM |
| Let agent read user preferences | 🔧 Tool call | LLM decides when it needs this info |
| Let agent update a database | 🔧 Tool call | LLM decides when to write |
| Let agent search the web | 🔧 Tool call | LLM decides when to search |

---

## 🎯 This notebook demonstrates all three side-by-side

We'll build the **same feature** — a user profile system — using each approach, so you can see exactly how they differ.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

---

## 🔀 Approach 1: Wrap-style Middleware

### "Adjust an agent's tools, prompt, model while running"

**Scenario**: Internal users get access to a database tool. External users only get web search. The LLM never knows tools were filtered — it just sees fewer tools.

**Key**: the **developer** makes this decision, not the LLM. The LLM can't override it.

In [ ]:
from dataclasses import dataclass
from typing import Callable
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage


# ── Two tools: one public, one restricted ──

@tool
def web_search(query: str) -> str:
    """Search the public web for information."""
    return f"Web results for: {query}"


@tool
def internal_database(sql: str) -> str:
    """Query the internal company database. Restricted to internal users."""
    return f"DB result: 42 rows matched for '{sql}'"


# ── Context: who is the user? ──

@dataclass
class UserProfile:
    user_role: str = "external"   # "internal" or "external"
    user_name: str = "Guest"


# ── Wrap-style middleware: filter tools by role ──

@wrap_model_call
def filter_tools_by_role(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """External users can only use web_search."""
    role = request.runtime.context.user_role

    if role != "internal":
        # Remove internal_database from the tools the LLM sees
        safe_tools = [t for t in request.tools if t.name != "internal_database"]
        request = request.override(tools=safe_tools)
        print(f"  🔀 [Middleware] External user → tools filtered to: {[t.name for t in safe_tools]}")
    else:
        print(f"  🔀 [Middleware] Internal user → all tools available")

    return handler(request)


agent_wrap = create_agent(
    model="gpt-5-nano",
    tools=[web_search, internal_database],
    context_schema=UserProfile,
    middleware=[filter_tools_by_role],
)

In [ ]:
# External user asks about the database → agent can't use it, falls back to web
print("=== External User ===")
response = agent_wrap.invoke(
    {"messages": [HumanMessage(content="How many customers do we have?")]},
    context=UserProfile(user_role="external", user_name="Bob"),
)
print(response["messages"][-1].content)

In [ ]:
# Internal user asks the same question → agent CAN use the database
print("=== Internal User ===")
response = agent_wrap.invoke(
    {"messages": [HumanMessage(content="How many customers do we have?")]},
    context=UserProfile(user_role="internal", user_name="Alice"),
)
print(response["messages"][-1].content)

### 🔍 What just happened?

The middleware **silently removed** `internal_database` from the tool list before the LLM saw it. The external user's LLM call literally didn't know that tool existed. This is **invisible to the LLM** — it's a developer-enforced policy.

---

## 🪝 Approach 2: Node-style Middleware

### "Adjust an agent's state while running"

**Scenario**: Before each LLM call, inject the user's name into the conversation as a system-level note. This modifies `messages[]` — the state — not the model or tools.

**Key**: this transforms the **state** (what the LLM reads), not the **configuration** (which model/tools).

In [ ]:
from langchain.agents.middleware import before_agent
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langchain.messages import SystemMessage
from typing import Any


@before_agent
def inject_user_context(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Add a system message with the user's name before each LLM call."""
    user_name = runtime.context.user_name
    
    # Check if we already injected (avoid duplicates across turns)
    for msg in state["messages"]:
        if isinstance(msg, SystemMessage) and "Current user:" in msg.content:
            return None  # already injected
    
    print(f"  🪝 [Node middleware] Injecting user context: {user_name}")
    return {
        "messages": [SystemMessage(content=f"Current user: {user_name}. Address them by name.")]
    }


agent_node = create_agent(
    model="gpt-5-nano",
    context_schema=UserProfile,
    middleware=[inject_user_context],
)

In [ ]:
response = agent_node.invoke(
    {"messages": [HumanMessage(content="What's the weather like today?")]},
    context=UserProfile(user_name="Alice"),
)
# The agent should address Alice by name
print(response["messages"][-1].content)

In [ ]:
response = agent_node.invoke(
    {"messages": [HumanMessage(content="What's the weather like today?")]},
    context=UserProfile(user_name="Carlos"),
)
# Now it should address Carlos
print(response["messages"][-1].content)

### 🔍 What just happened?

The middleware **modified the state** by adding a `SystemMessage` to `messages[]`. The LLM saw this message and used it to personalize the response. Unlike wrap-style, this doesn't change which model or tools are used — it changes **what the LLM reads**.

---

## 🔧 Approach 3: Tool Calls

### "Agent adjusts its own state or retrieves runtime context"

**Scenario**: The agent has a tool to look up the user's name from context, and a tool to save preferences to state. The **LLM decides** when to call these tools — not the developer.

**Key**: the LLM is in control. It sees the tool descriptions and chooses when to use them.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.messages import ToolMessage


# ── Custom state to track preferences ──

from langchain.agents import AgentState as BaseAgentState

class PreferenceState(BaseAgentState):
    favourite_color: str


# ── Tool: read user name from context ──

@tool
def get_user_name(runtime: ToolRuntime) -> str:
    """Get the current user's name."""
    name = runtime.context.user_name
    print(f"  🔧 [Tool] LLM called get_user_name → {name}")
    return name


# ── Tool: save preference to state ──

@tool
def save_favourite_color(color: str, runtime: ToolRuntime) -> Command:
    """Save the user's favourite color to the agent's memory."""
    print(f"  🔧 [Tool] LLM called save_favourite_color → {color}")
    return Command(
        update={
            "favourite_color": color,
            "messages": [
                ToolMessage(
                    content=f"Saved favourite color: {color}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# ── Tool: read preference from state ──

@tool
def get_favourite_color(runtime: ToolRuntime) -> str:
    """Read the user's favourite color from memory."""
    color = runtime.state.get("favourite_color", "not set yet")
    print(f"  🔧 [Tool] LLM called get_favourite_color → {color}")
    return color


agent_tools = create_agent(
    model="gpt-5-nano",
    tools=[get_user_name, save_favourite_color, get_favourite_color],
    context_schema=UserProfile,
    state_schema=PreferenceState,
    checkpointer=InMemorySaver(),
)

In [ ]:
# The LLM decides to call get_user_name to learn who it's talking to
print("=== Turn 1: Who am I? ===")
response = agent_tools.invoke(
    {"messages": [HumanMessage(content="Hi! What's my name?")]},
    {"configurable": {"thread_id": "demo-1"}},
    context=UserProfile(user_name="Alice"),
)
print(response["messages"][-1].content)

In [ ]:
# The LLM decides to call save_favourite_color
print("\n=== Turn 2: Save a preference ===")
response = agent_tools.invoke(
    {"messages": [HumanMessage(content="My favourite color is teal")]},
    {"configurable": {"thread_id": "demo-1"}},
    context=UserProfile(user_name="Alice"),
)
print(response["messages"][-1].content)
print("\nState:", response.get("favourite_color", "(not set)"))

In [ ]:
# The LLM decides to call get_favourite_color to recall it
print("\n=== Turn 3: Recall the preference ===")
response = agent_tools.invoke(
    {"messages": [HumanMessage(content="What's my favourite color?")]},
    {"configurable": {"thread_id": "demo-1"}},
    context=UserProfile(user_name="Alice"),
)
print(response["messages"][-1].content)

### 🔍 What just happened?

The **LLM decided** when to call each tool. We didn't force it — we gave it tools with clear descriptions, and it chose to:
1. Call `get_user_name` when asked "What's my name?"
2. Call `save_favourite_color` when the user said "My favourite color is teal"
3. Call `get_favourite_color` when asked to recall it

This is fundamentally different from middleware — the LLM is **in the driver's seat**.

---

## ⚖️ Side-by-Side Comparison

Let's compare what happened in each approach:

| Aspect | 🔀 Wrap-style | 🪝 Node-style | 🔧 Tool Calls |
|---|---|---|---|
| **Who decides** | Developer (you) | Developer (you) | LLM (the agent) |
| **LLM aware?** | ❌ No — tools silently filtered | ❌ No — messages silently modified | ✅ Yes — LLM sees and calls tools |
| **What changes** | Model, tools, prompt | State (messages, custom fields) | State (via Command) or reads context |
| **Receives** | `ModelRequest` | `AgentState` + `Runtime` | `ToolRuntime` |
| **Runs when** | Around each LLM/tool call | Before/after graph nodes | When LLM decides |
| **Can be bypassed?** | ❌ No — always runs | ❌ No — always runs | ✅ Yes — LLM might not call it |

---

## 🧠 The Mental Model

Think of an agent as a person working at a desk:

- **🔀 Wrap-style middleware** = the **IT department** silently configuring their computer. They decide which software is installed, which websites are accessible. The person at the desk doesn't know what was blocked.

- **🪝 Node-style middleware** = the **office manager** who organizes their inbox before they start working. Old emails get archived, important ones get flagged. The person sees a clean inbox but didn't do the organizing.

- **🔧 Tool calls** = the **tools on their desk** — a phone, a calculator, a filing cabinet. The person **chooses** when to pick up the phone or open the cabinet. Nobody forces them.

---

## 🎯 Decision Flowchart

```
Should the LLM decide when this happens?
  ├── YES → Use a Tool Call (@tool with ToolRuntime)
  └── NO → Is it about which model/tools/prompt to use?
              ├── YES → Use Wrap-style (@wrap_model_call, @dynamic_prompt)
              └── NO → Use Node-style (@before_agent, SummarizationMiddleware)
```

---

## 💡 Can I combine them?

Absolutely. In fact, production agents typically use all three:

```python
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_user_name, save_preference, web_search],  # 🔧 Tool calls
    middleware=[
        filter_tools_by_role,      # 🔀 Wrap-style: RBAC
        SummarizationMiddleware(), # 🪝 Node-style: manage history
        language_prompt,           # 🔀 Config-style: dynamic prompt
    ],
    context_schema=UserProfile,
    state_schema=PreferenceState,
    checkpointer=InMemorySaver(),
)
```

They compose naturally because they operate at different levels:
1. Wrap-style configures the LLM call
2. Node-style cleans up the state
3. Tools give the LLM capabilities

No conflicts. Each does its job.